In [92]:
from pathlib import Path
import json
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer

In [93]:
BASE_DIR = Path.cwd()
if (BASE_DIR / 'data' / 'sft_t2t_mini.jsonl').exists():
    BASE_DIR = BASE_DIR
elif (BASE_DIR.parent / 'data' , 'sft_t2t_mini.jsonl'):
    BASE_DIR = BASE_DIR.parent
else:
    BASE_DIR = BASE_DIR

DATA_PATH = BASE_DIR / 'data' / 'sft_t2t_mini.jsonl'
TOKENIZER_DIR = BASE_DIR / 'tokenizer'

VOCAB_SIZE = 6400

SPECIAL_TOKENS_NUM = 36

In [94]:
with open(DATA_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if i >= 1000:
            break
        try:
            data = json.loads(line)
            print(line, data)

            contents = [
                item.get('content') 
                for item in data.get("conversations", []) 
                if item.get('content')
            ]
            print(contents)
            print("\n".join(contents))
        except json.JSONDecodeError:
            continue

def get_texts(data_path):
    with open(DATA_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i >= 1000:
                break
            try:
                data = json.loads(line)
                # print(line, data)
                contents = [
                    item.get('content') 
                    for item in data.get("conversations", []) 
                    if item.get('content')
                ]
                if contents:
                    yield "\n".join(contents)
            except json.JSONDecodeError:
                continue


{"conversations":[{"role":"user","content":"你好，请用一句话介绍什么是 tokenizer。"},{"role":"assistant","content":"tokenizer 是把文本切分成 token，并在文本和 token id 之间转换的工具。"}]}
 {'conversations': [{'role': 'user', 'content': '你好，请用一句话介绍什么是 tokenizer。'}, {'role': 'assistant', 'content': 'tokenizer 是把文本切分成 token，并在文本和 token id 之间转换的工具。'}]}
['你好，请用一句话介绍什么是 tokenizer。', 'tokenizer 是把文本切分成 token，并在文本和 token id 之间转换的工具。']
你好，请用一句话介绍什么是 tokenizer。
tokenizer 是把文本切分成 token，并在文本和 token id 之间转换的工具。
{"conversations":[{"role":"user","content":"把下面这句话切成更容易理解的小块：我正在学习大语言模型的分词器训练。"},{"role":"assistant","content":"可以切成：我 / 正在 / 学习 / 大语言模型 / 的 / 分词器 / 训练。真实 tokenizer 会根据词表和算法得到不同切分。"}]}
 {'conversations': [{'role': 'user', 'content': '把下面这句话切成更容易理解的小块：我正在学习大语言模型的分词器训练。'}, {'role': 'assistant', 'content': '可以切成：我 / 正在 / 学习 / 大语言模型 / 的 / 分词器 / 训练。真实 tokenizer 会根据词表和算法得到不同切分。'}]}
['把下面这句话切成更容易理解的小块：我正在学习大语言模型的分词器训练。', '可以切成：我 / 正在 / 学习 / 大语言模型 / 的 / 分词器 / 训练。真实 tokenizer 会根据词表和算法得到不同切分。']
把下面这句话切成更容易理解的小块：我正在学习大语言模型的分词器训练。
可以切成：

In [95]:
models.BPE()

BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[])

In [96]:
# 只是一个空的 BPE 模型，还没有学会任何词表。
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
special_tokens_list = [
    "<|endoftext|>",
    "<|im_start|>",
    "<|im_end|>",
    "<|object_ref_start|>",
    "<|object_ref_end|>",
    "<|box_start|>",
    "<|box_end|>",
    "<|quad_start|>",
    "<|quad_end|>",
    "<|vision_start|>",
    "<|vision_end|>",
    "<|vision_pad|>",
    "<|image_pad|>",
    "<|video_pad|>",
    "<|audio_start|>",
    "<|audio_end|>",
    "<|audio_pad|>",
    "<tts_pad>",
    "<tts_text_bos>",
    "<tts_text_eod>",
    "<tts_text_bos_single>",
]

additional_tokens_list = [
    "<tool_call>",
    "</tool_call>",
    "<tool_response>",
    "</tool_response>",
    "<think>",
    "</think>",
]

In [97]:
num_buffer = SPECIAL_TOKENS_NUM - len(special_tokens_list + additional_tokens_list)
buffer_tokens = [f"<buffer{i}>" for i in range(1, num_buffer + 1)]
all_special_tokens = special_tokens_list + additional_tokens_list + buffer_tokens

In [98]:
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    special_tokens = all_special_tokens
)

In [99]:
texts = get_texts(DATA_PATH)
texts

<generator object get_texts at 0x1075c70a0>

In [100]:
tokenizer.train_from_iterator(texts, trainer=trainer)


In [101]:
tokenizer.decoder = decoders.ByteLevel()
tokenizer.add_special_tokens(special_tokens_list)


0

In [102]:
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tokenizer.save(str(TOKENIZER_DIR / 'tokenizer.json'))
tokenizer.model.save(str(TOKENIZER_DIR))

['/Users/lanjie/MLLM/minimind/tokenizer/vocab.json',
 '/Users/lanjie/MLLM/minimind/tokenizer/merges.txt']

In [103]:
TOKENIZER_JSON_PATH = TOKENIZER_DIR / 'tokenizer.json'

In [104]:
with open(TOKENIZER_JSON_PATH, 'r', encoding='utf-8') as f:
    tokenizer_data = json.load(f)

In [105]:
for token_info in tokenizer_data.get('added_tokens', []):
    if token_info['content'] not in special_tokens_list:
        token_info['special'] = False

In [106]:
with open(TOKENIZER_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(tokenizer_data, f, ensure_ascii=False, indent=2)

In [ ]:
added_tokens_decoder = {}
for token in all_special_tokens:
    idx = tokenizer.token_to_id(token)
    added_tokens_decoder[str[idx]] = {
        "content": token,
        "lstrip": False,
        'normalized': False,
        'rstrip': False,
        'single_word': False,
        'special': True if token in special_tokens_list else False,
    }

In [ ]:
config = {
    "add_bos_token": False,
    "add_eos_token": False,
    "add_prefix_space": False,
    "added_tokens_decoder": added_tokens_decoder,
    "additional_special_tokens": [
        t for t in special_tokens_list if t not in ["<|endoftext|>"]
    ],
    "bos_token": "<|im_start|>",
    "clean_up_tokenization_spaces": False,
    "eos_token": "<|im_end|>",
    "legacy": True,
    "model_max_length": 131072,
    "pad_token": "<|endoftext|>",
    "sp_model_kwargs": {},
    "spaces_between_special_tokens": False,
    "unk_token": "<|endoftext|>",
    "image_token": "<|image_pad|>",
    "audio_token": "<|audio_pad|>",
    "video_token": "<|video_pad|>",
    "vision_bos_token": "<|vision_start|>",
    "vision_eos_token": "<|vision_end|>",
    "audio_bos_token": "<|audio_start|>",
    "audio_eos_token": "<|audio_end|>",
    "chat_template": "{%- if tools %}\n    {{- '<|im_start|>system\\n' }}\n    {%- if messages[0].role == 'system' %}\n        {{- messages[0].content + '\\n\\n' }}\n    {%- endif %}\n    {{- \"# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0].role == 'system' %}\n        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}\n    {%- endif %}\n{%- endif %}\n{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}\n{%- for message in messages[::-1] %}\n    {%- set index = (messages|length - 1) - loop.index0 %}\n    {%- if ns.multi_step_tool and message.role == \"user\" and message.content is string and not(message.content.startswith('<tool_response>') and message.content.endswith('</tool_response>')) %}\n        {%- set ns.multi_step_tool = false %}\n        {%- set ns.last_query_index = index %}\n    {%- endif %}\n{%- endfor %}\n{%- for message in messages %}\n    {%- if message.content is string %}\n        {%- set content = message.content %}\n    {%- else %}\n        {%- set content = '' %}\n    {%- endif %}\n    {%- if (message.role == \"user\") or (message.role == \"system\" and not loop.first) %}\n        {{- '<|im_start|>' + message.role + '\\n' + content + '<|im_end|>' + '\\n' }}\n    {%- elif message.role == \"assistant\" %}\n        {%- set reasoning_content = '' %}\n        {%- if message.reasoning_content is string %}\n            {%- set reasoning_content = message.reasoning_content %}\n        {%- else %}\n            {%- if '</think>' in content %}\n                {%- set reasoning_content = content.split('</think>')[0].rstrip('\\n').split('<think>')[-1].lstrip('\\n') %}\n                {%- set content = content.split('</think>')[-1].lstrip('\\n') %}\n            {%- endif %}\n        {%- endif %}\n        {%- if true %}\n            {{- '<|im_start|>' + message.role + '\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' + content.lstrip('\\n') }}\n        {%- endif %}\n        {%- if message.tool_calls %}\n            {%- for tool_call in message.tool_calls %}\n                {%- if (loop.first and content) or (not loop.first) %}\n                    {{- '\\n' }}\n                {%- endif %}\n                {%- if tool_call.function %}\n                    {%- set tool_call = tool_call.function %}\n                {%- endif %}\n                {{- '<tool_call>\\n{\"name\": \"' }}\n                {{- tool_call.name }}\n                {{- '\", \"arguments\": ' }}\n                {%- if tool_call.arguments is string %}\n                    {{- tool_call.arguments }}\n                {%- else %}\n                    {{- tool_call.arguments | tojson }}\n                {%- endif %}\n                {{- '}\\n</tool_call>' }}\n            {%- endfor %}\n        {%- endif %}\n        {{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}\n        {%- if loop.first or (messages[loop.index0 - 1].role != \"tool\") %}\n            {{- '<|im_start|>user' }}\n        {%- endif %}\n        {{- '\\n<tool_response>\\n' }}\n        {{- content }}\n        {{- '\\n</tool_response>' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}\n            {{- '<|im_end|>\\n' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- '<|im_start|>assistant\\n' }}\n    {%- if open_thinking is defined and open_thinking is true %}\n        {{- '<think>\\n' }}\n    {%- else %}\n        {{- '<think>\\n\\n</think>\\n\\n' }}\n    {%- endif %}\n{%- endif %}",
    "tokenizer_class": "PreTrainedTokenizerFast",
}

# 写出 Hugging Face 风格配置后，这个目录就能被 `AutoTokenizer.from_pretrained(...)`
# 直接加载，用法会和常见开源模型保持一致。
with open(
    os.path.join(tokenizer_dir, "tokenizer_config.json"), "w", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=4)
print("Tokenizer training completed.")